# basic 

In [31]:
%load_ext autoreload
%autoreload all

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [32]:
import polars as pl
import pickle


import src.configs_new_mapped as configs_mapped
import src.graph_fct as graph_fct

# 1. prepare mapped concepts and subgraph

In [33]:
df_relations = pl.read_parquet(f"{configs_mapped.GraphConfig().relation_path}")
df_mapped = pl.read_parquet(f"{configs_mapped.GraphConfig().concept_path}").filter(pl.col("is_mapped")).unique()

In [34]:
mapped_ids = df_mapped["id"].to_list()
whole_graph = graph_fct.build_relations_graph(df_relations, col_src="src.id", col_dst="dst.id", col_relation="relation")
combined_subgraphs = graph_fct.get_combined_subgraphs_from_nodes(whole_graph, mapped_ids, max_distance=configs_mapped.TokenizerParam().max_dist_candidate)

In [35]:
len(combined_subgraphs.nodes()), len(combined_subgraphs.edges()), list(combined_subgraphs.edges(data=True))[:5]



(75635,
 290762,
 [('113343008', '91723000', {'relation': 'IS_A'}),
  ('312336005', '129284003', {'relation': 'IS_A'}),
  ('182353008', '272424004', {'relation': 'IS_A'}),
  ('13714004+449363000', '129433002', {'relation': 'METHOD'}),
  ('13714004+449363000', '129304002', {'relation': 'METHOD'})])

In [36]:
# save
with open(configs_mapped.ProcessedGraph().combined_subgraphs, "wb") as f:
    pickle.dump(combined_subgraphs, f)

df_mapped.write_parquet(configs_mapped.ProcessedGraph().mapped_cpts)

# tokenizer

In [47]:
import itertools

class Context:
    """One node of a context tree.
    ISA edges stay in the current context; only non-ISA edges open a sub-context."""

    def __init__(self, labels, relation="ROOT", distance=0):
        self.labels      = labels      # shared id -> label dict
        self.relation    = relation
        self.distance    = distance
        self.destination = None        # concept this subcontext points at (None for ROOT)
        self.tokens      = []          # [(token, distance)]  <-- single source of truth
        self.uncovered   = False
        self.subcontexts = []

    def _label(self, cid):
        return self.labels.get(cid, cid)   # fall back to the id if not in the map

    def add_token(self, token, distance):
        self.tokens.append((token, distance))

    def mark_uncovered(self):
        self.uncovered = True

    def open_subcontext(self, relation, destination, distance):
        for child in self.subcontexts:
            if child.relation == relation and child.destination == destination:
                child.distance = min(child.distance, distance)   # keep most direct
                return child, False
        child = Context(self.labels, relation, distance)          # <-- pass labels through
        child.destination = destination
        self.subcontexts.append(child)
        return child, True
    
    def to_sexpr(self, show_uncovered=True):
        items  = [str(tok) for tok, _ in self.tokens]
        if self.uncovered and show_uncovered:
            items.append("∅")
        items += [f"{sub.relation}({sub.to_sexpr(show_uncovered)})"
                  for sub in self.subcontexts]
        return ", ".join(items)

    def __repr__(self):
        return f"({self.to_sexpr()})"
    # def __repr__(self):
    #     counter = itertools.count()
    #     def render(ctx, indent):
    #         qid, pad = f"q{next(counter)}", "    " * indent
    #         head = ctx.relation
    #         if ctx.destination is not None:                       # show what the edge points at
    #             head += f" → {ctx.destination} ({ctx._label(ctx.destination)})"
    #         lines = [f"{pad}[{qid}] {head}"]
    #         for tok, d in ctx.tokens:
    #             if tok == ctx.destination:            # already shown in the header
    #                 lines.append(f"{pad}    token: {tok}  (d={d})")
    #             else:
    #                 lines.append(f"{pad}    token: {tok} ({ctx._label(tok)})  (d={d})")
    #         if ctx.uncovered:
    #             lines.append(f"{pad}    uncovered")
    #         for sub in ctx.subcontexts:
    #             lines.append(render(sub, indent + 1))
    #         return "\n".join(lines)
    #     return render(self, 0)

def signature(ctx):
    """Content identity of a context, ignoring distance."""
    return (
        ctx.relation,
        ctx.uncovered,
        frozenset(tok for tok, _ in ctx.tokens),
        frozenset(signature(sub) for sub in ctx.subcontexts),
    )

def collapse_redundant(ctx):
    for sub in ctx.subcontexts:          # recurse first: resolve nested dups bottom-up
        collapse_redundant(sub)

    best = {}                            # signature -> shortest-distance subcontext
    for sub in ctx.subcontexts:
        key = signature(sub)
        if key not in best or sub.distance < best[key].distance:
            best[key] = sub
    ctx.subcontexts = list(best.values())   # dict preserves first-seen order

    seen = {}                            # same idea for tokens in THIS context
    for tok, dd in ctx.tokens:
        if tok not in seen or dd < seen[tok]:
            seen[tok] = dd
    ctx.tokens = list(seen.items())
    return ctx
    

In [48]:
def tokenize(c, G, T, D, id_to_label):
    root = Context(id_to_label, "ROOT", distance=0)
    expand(c, root, 0, G, T, D)
    # return collapse_redundant(root)
    return root

def expand(u, q, d, G, T, D):
    if u in T:
        q.add_token(u, d)
    elif d == D or G.out_degree(u) == 0:
        q.mark_uncovered()
    else:
        # non-IS_A first: the direct (shallower) statement wins; parents' restatements dedup away
        edges = sorted(G.out_edges(u, data=True),
                       key=lambda e: e[2]["relation"] == "IS_A")
        for _, v, data in edges:
            r = data["relation"]
            if r == "IS_A":
                expand(v, q, d + 1, G, T, D)          # transparent, stay in q
            else:
                child, is_new = q.open_subcontext(r, v, d + 1)
                if is_new:
                    expand(v, child, d + 1, G, T, D)

In [39]:
with open(configs_mapped.ProcessedGraph().combined_subgraphs, "rb") as f:
    combined_subgraphs = pickle.load(f)
df = pl.read_parquet(configs_mapped.GraphConfig().concept_path).select("id", "label")
id_to_label = dict(zip(df["id"], df["label"]))

In [ ]:
tokenize("392091000", combined_subgraphs, ["225365006", "128927009", "129265001"], 3, id_to_label)
# "386053000"

(128927009, HAS_FOCUS(225365006), METHOD(129265001))

In [50]:
tokenize("392091000", combined_subgraphs, ["225365006", "129265001"], 3, id_to_label)


(∅, HAS_FOCUS(225365006), METHOD(129265001), METHOD(∅))

In [52]:
tokenize("392091000", combined_subgraphs, ["392091000", "128927009"],3, id_to_label)


(392091000)

In [53]:
tokenize("95436008", combined_subgraphs, ["182353008"],3,id_to_label)


(∅, FINDING_SITE(∅, LATERALITY(182353008)), ASSOCIATED_MORPHOLOGY(∅), FINDING_SITE(∅), FINDING_SITE(∅))

# semantic coverage compuation

In [77]:
import src.propagate as propagate
sinks = {n for n in combined_subgraphs.nodes() if combined_subgraphs.out_degree(n) == 0}
flags = {cpt: 0 for cpt in sinks}

for cpt in ["58160005","197613008", "18602000", "64033007", "2123749006", "90734009"]:
    flags[cpt] = 1

edges = list(combined_subgraphs.edges())


result = propagate.propagate_n_get_value(edges, flags, verbose=True)


  363935009 = 0
  244023005 = 0
  399797005 = 0
  299982002 = 0
  426722004 = 0
  414468001 = 0
  699379009 = 0
  364207009 = 0
  409840004 = 0
  425572006 = 0
  773580005 = 0
  407455007 = 0
  281727004 = 0
  414200008 = 0
  106723004 = 0
  417487004 = 0
  363859000 = 0
  418920007 = 0
  425807007 = 0
  277565005 = 0
  115178001 = 0
  255922001 = 0
  229315006 = 0
  284488007 = 0
  91690000 = 0
  363992005 = 0
  426765003 = 0
  72353004 = 0
  364483009 = 0
  723987002 = 0
  279507009 = 0
  279584000 = 0
  399949004 = 0
  120669005 = 0
  427005001 = 0
  414961002 = 0
  705330007 = 0
  705731009 = 0
  53982002 = 0
  415656009 = 0
  364349001 = 0
  384740007 = 0
  426663000 = 0
  1290206009 = 0
  1360041005 = 0
  736680000 = 0
  417503000 = 0
  46815006 = 0
  228780002 = 0
  709869000 = 0
  281719008 = 0
  785315000 = 0
  86308005 = 0
  871491005 = 0
  224611006 = 0
  767036003 = 0
  406758004 = 0
  703725008 = 0
  72869002 = 0
  228955007 = 0
  308339002 = 0
  282970009 = 0
  284349006 